# Advanced Deep Learning - Exercise 1 - Introduction

In this exercise, we will learn the basics of training a neural network with PyTorch. Specifically, we will train a convolutional neural network for image classification on the CIFAR-10 dataset. While doing so, we will also revisit the basics of experiment tracking and management.

First, let's install some dependencies. We will use wandb for logging and plotly for visualization.

In [ ]:
%pip install wandb
%pip install plotly

The following command should be run only once. Please comment it or delete the following cell after initializing Weights & Biases

In [ ]:
#!timeout 5m wandb init

In [ ]:
# Import all necessary packages
import copy
from typing import Any

import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.transforms import v2

import wandb

In [ ]:
wandb.login()

The following is a helper to properly visualize data using plotly in notebooks/colab.

In [ ]:
def configure_plotly_browser_state():
    """
    In Google Colab, this function needs to be called before each plot in each
    cell that contains a plot. Otherwise, things will not be displayed properly.
    """
    import IPython

    display(IPython.core.display.HTML('''
        <script src="/static/components/requirejs/require.js"></script>
        <script>
          requirejs.config({
            paths: {
              base: '/static/base',
              plotly: 'https://cdn.plot.ly/plotly-latest.min.js?noext',
            },
          });
        </script>
        '''))
  

def plotly_display(fig):
    plotly.offline.init_notebook_mode()
    configure_plotly_browser_state()
    plotly.offline.iplot(fig)

It is generally a good idea to manage a config dictionary that contains all your hyperparameters. You can even keep track of this using wandb

In [ ]:
default_config = {
    "project": "adl26_exercise_01",
    "experiment": "MyFirstExperiment",
    "batch_size_train": 256,
    "batch_size_val": 256,
    "epochs": 5,
    "lr": 0.001,
    "momentum": 0.9,
    "seed": 42,
    "log_interval": 10,
}

default_device = "cuda" if torch.cuda.is_available() else "cpu"


def get_config(**kwargs) -> dict[str, Any]:
    config = copy.deepcopy(default_config)
    config.update(kwargs)
    return config

Now let's set up our data. We will be using CIFAR 10.

In [ ]:
data_root = "./data"

# Define train and test transformation.
train_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

test_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

# Define train and test datasets and data loaders
train_dataset = datasets.CIFAR10(
    data_root, train=True, download=True, transform=train_transform
)

test_dataset = datasets.CIFAR10(data_root, train=False, transform=test_transform)


# Get teh classes defined in the datasets
classes = train_dataset.classes


def setup_dataloaders(
    config: dict[str, Any],
    train_dataset: Dataset,
    val_dataset: Dataset,
    device: torch.device, **kwargs
) -> tuple[DataLoader, DataLoader]:
    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size_train"],
        generator=torch.Generator(device=device),
        shuffle=True,
        **kwargs,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size_val"],
        generator=torch.Generator(device=device),
        shuffle=False,
        **kwargs,
    )

    return train_loader, val_loader

These are all the classes that we use:

In [ ]:
classes

We can visualize some examples:

In [ ]:
def visualize_samples():
    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=[
            f"Ground Truth: {classes[train_dataset[i][1]]}" for i in range(6)
        ],
    )

    for i in range(6):
        example_data, example_targets = train_dataset[i]
        img = example_data.permute(1, 2, 0).cpu().numpy() / 2 + 0.5  # Unnormalize

        row = i // 3 + 1
        col = i % 3 + 1

        fig.add_trace(go.Image(z=(img * 255).astype("uint8")), row=row, col=col)

    # Hide axis ticks
    for i in range(1, 7):
        fig.update_xaxes(
            showticklabels=False, row=(i - 1) // 3 + 1, col=(i - 1) % 3 + 1
        )
        fig.update_yaxes(
            showticklabels=False, row=(i - 1) // 3 + 1, col=(i - 1) % 3 + 1
        )

    fig.update_layout(
        height=600,
        width=900,
        showlegend=False,
        title_text="Example Images with Ground Truth Labels",
    )

    plotly_display(fig)


visualize_samples()

We can now define a basic network:

In [ ]:
class Net(nn.Module):
    # This defines the structure of the NN.
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(32 * 32 * 3, 10)

    def forward(self, x):
        # Reshape and treat pixels as channels
        x = x.view(-1, 32 * 32 * 3)

        # Fully Connected Layer
        x = self.fc1(x)

        return x

And some helper functions to train it:

In [ ]:
def train_one_epoch(
    model: nn.Module,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    dataloader: DataLoader,
    epoch: int,
    log_interval: int,
    device: torch.device,
) -> tuple[list[int], list[float]]:
    # Make sure that the model is in training mode
    model.train()

    log_losses = []
    log_steps = []

    for batch_idx, (inputs, target) in enumerate(dataloader):
        # Move the inputs and targets to the device
        inputs = inputs.to(device)
        target = target.to(device)

        # Clear/reset the gradient buffers
        optimizer.zero_grad()

        # Run the forward pass
        preds = model(inputs)

        # Compute the loss
        loss = criterion(preds, target).mean()

        # Run the backward pass and compute gradients
        loss.backward()

        # Perform one optimizer step
        optimizer.step()

        # Once in a while: Do some logging and collect losses
        if batch_idx % log_interval == 0:
            batch_size = inputs.shape[0]

            steps = batch_idx * batch_size
            num_steps_per_epoch = len(dataloader.dataset)

            print(
                "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                    epoch,
                    steps,
                    num_steps_per_epoch,
                    100.0 * batch_idx / len(dataloader),
                    loss.item(),
                )
            )

            total_steps = (epoch - 1) * num_steps_per_epoch + steps
            log_steps.append(total_steps)
            log_losses.append(loss.item())

            wandb.log({"train_loss": loss.item()}, step=total_steps)

    return log_steps, log_losses

In [ ]:
def validate_one_epoch(
    model: nn.Module,
    criterion: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
) -> tuple[float, float]:
    # Make sure that the model is in eval mode
    model.eval()

    total_loss = 0
    total_correct = 0

    with torch.no_grad():
        for inputs, target in dataloader:
            # Move the inputs and targets to the device
            inputs = inputs.to(device)
            target = target.to(device)

            # Run the forward pass / compute the predictions
            preds = model(inputs)

            # Compute and update the loss
            loss = criterion(preds, target).sum()
            total_loss += loss.item()

            # Compute the actual classification result as argmax (index with maximum confidence)
            result = preds.max(dim=1, keepdim=True)[1]

            # Count the number of correct predictions and update the total count
            correct = (result == target.view_as(result)).sum()
            total_correct += correct.item()

    # Average the loss over all samples and compute average accuracy
    num_samples = len(dataloader.dataset)
    total_loss = total_loss / num_samples
    accuracy = 100.0 * total_correct / num_samples

    print(
        "\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n".format(
            total_loss, total_correct, num_samples, accuracy
        )
    )

    # Log the metrics with wandb
    wandb.log(
        {
            "test_loss": total_loss,
            "test_accuracy": accuracy,
        },
        commit=False,
    )

    return total_loss, accuracy

In [ ]:
def train(
    model: nn.Module,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    dataloader_train: DataLoader,
    dataloader_val: DataLoader,
    device: torch.device,
    num_epochs: int,
    log_interval: int,
):
    train_steps = []
    train_losses = []

    val_steps = [i * len(dataloader_train.dataset) for i in range(num_epochs + 1)]
    val_losses = []
    val_accuracies = []

    val_loss, val_acc = validate_one_epoch(model, criterion, dataloader_val, device)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    for epoch in range(1, num_epochs + 1):
        epoch_steps, epoch_losses = train_one_epoch(
            model,
            optimizer,
            criterion,
            dataloader_train,
            epoch,
            log_interval=log_interval,
            device=device,
        )
        train_steps += epoch_steps
        train_losses += epoch_losses

        val_loss, val_acc = validate_one_epoch(model, criterion, dataloader_val, device)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

    return train_steps, train_losses, val_steps, val_losses, val_accuracies

This is a generic helper function that given a model, some data, and some optional config arguments trains our model and returns some logged values like losses and accuracies over time.

In [ ]:
def train_model(
    model: nn.Module,
    dataset_train: Dataset = train_dataset,
    dataset_val: Dataset = test_dataset,
    device: torch.device = default_device,
    **kwargs,
):
    config = get_config(**kwargs)

    wandb.init(config=config, project=config["project"], name=config["experiment"])

    # Set the default device
    torch.set_default_device(device)

    # Move the module to the device
    model = model.to(device)

    # Create the optimizer and loss function
    optimizer = optim.SGD(
        model.parameters(), lr=config["lr"], momentum=config["momentum"]
    )
    criterion = nn.CrossEntropyLoss(reduction="none")

    # Set up the dataloaders
    dataloader_train, dataloader_val = setup_dataloaders(
        config, train_dataset=dataset_train, val_dataset=dataset_val, device=device
    )

    log_data = train(
        model,
        optimizer,
        criterion,
        dataloader_train,
        dataloader_val,
        device,
        num_epochs=config["epochs"],
        log_interval=config["log_interval"],
    )

    # Finish the run
    wandb.finish()

    return log_data

Let's now use that to train the model:

In [ ]:
model = Net()

logs = train_model(model)

We can also visualize the metrics here:

In [ ]:
def visualize_metrics(train_steps, train_losses, val_steps, val_losses, val_accuracies):
    df_loss = pd.DataFrame(
        {
            "Step": train_steps + val_steps,
            "Loss": train_losses + val_losses,
            "Type": ["Train"] * len(train_steps) + ["Test"] * len(val_steps),
        }
    )

    fig_loss = px.line(
        df_loss,
        x="Step",
        y="Loss",
        color="Type",
        title="Train and Test Loss",
        labels={
            "Step": "Number of Training Examples Seen",
            "Loss": "Cross Entropy Loss",
        },
    )
    df_acc = pd.DataFrame({"Step": val_steps, "Accuracy": val_accuracies})

    fig_acc = px.line(
        df_acc,
        x="Step",
        y="Accuracy",
        title="Test Accuracy",
        labels={"Step": "Number of Training Examples Seen", "Accuracy": "Accuracy (%)"},
    )
    fig_acc.update_yaxes(range=[0, 100])

    plotly_display(fig_loss)
    plotly_display(fig_acc)

    print(f"Final metrics on test set: Accuracy: {val_accuracies[-1]}")

In [ ]:
visualize_metrics(*logs)

And some predictions:

In [ ]:
@torch.no_grad
def visualize_predictions(
    model: nn.Module,
    data: Dataset = test_dataset,
    device: torch.device = default_device,
):
    loader = DataLoader(data, batch_size=default_config["batch_size_val"])

    examples = enumerate(loader)
    batch_idx, (example_data, example_targets) = next(examples)
    output = model(example_data.to(device))

    # Get predictions
    preds = output.cpu().data.max(1, keepdim=True)[1].squeeze().tolist()

    # Prepare subplot layout: 2 rows, 3 columns
    fig = make_subplots(
        rows=2,
        cols=3,
        subplot_titles=[f"Prediction: {classes[preds[i]]}" for i in range(6)],
    )

    for i in range(6):
        img = example_data[i].permute(1, 2, 0).cpu().numpy() / 2 + 0.5  # unnormalize
        row = i // 3 + 1
        col = i % 3 + 1
        fig.add_trace(go.Image(z=(img * 255).astype("uint8")), row=row, col=col)

    # Hide axis ticks for all subplots
    for i in range(1, 7):
        fig.update_xaxes(
            showticklabels=False, row=(i - 1) // 3 + 1, col=(i - 1) % 3 + 1
        )
        fig.update_yaxes(
            showticklabels=False, row=(i - 1) // 3 + 1, col=(i - 1) % 3 + 1
        )

    fig.update_layout(height=600, width=900, showlegend=False, title_text="Predictions")

    plotly_display(fig)

In [ ]:
visualize_predictions(model)

## Questions:

- What can you see?
- What is "wrong"?

Add your additional experiments and attempts below. You can define a new model, augment the training data, or change hyperparameters like the learning rate. Then just run
```python
model = Net()

logs = train_model(model, ...any additional arguments like learning-rate here...)
```
again and optionally
```python
visualize_predictions(model)
```
to visualize your results directly in the notebook.